In [83]:
import os
import glob
from pathlib import Path

current_dir = Path.cwd()
data_dir = current_dir.parent/'data'
processed_dir = current_dir.parent/'processed_data'
os.makedirs(processed_dir, exist_ok=True)
supporting_dir = current_dir.parent/'supporting_data'
viz_dir = current_dir.parent/'visualizations'
os.makedirs(viz_dir, exist_ok=True)

In [84]:
import pandas as pd
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 25)
import numpy as np
import re
import json
import matplotlib.pyplot as plt
import seaborn as sns

import geopandas as gpd
from shapely.geometry import Point, shape
import folium
from folium.utilities import JsCode
from folium.features import GeoJsonPopup
from folium.plugins.timeline import Timeline, TimelineSlider

In [85]:
gdf_list = []

for file in glob.glob(f'{data_dir}/*.geojson'):
    file_year = ''.join(re.findall(r'\d+', str(file)))
    gdf= gpd.read_file(file)
    gdf['map_year'] = file_year
    gdf.columns = gdf.columns.str.replace('__', '_')
    gdf_list.append(gdf)

raw_gdf= pd.concat(gdf_list)
raw_gdf.columns= raw_gdf.columns.str.lower()
"""gdf= gdf[['gang_name', 'map_year', 'geometry']]"""
raw_gdf = raw_gdf.map(lambda x: x.lower() if isinstance(x, str) else x)
raw_gdf['map_year'] = raw_gdf['map_year'].astype(int)

raw_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1052 entries, 0 to 50
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   objectid      862 non-null    float64 
 1   gang_name     1050 non-null   object  
 2   shape_area    732 non-null    float64 
 3   shape_length  638 non-null    float64 
 4   geometry      988 non-null    geometry
 5   map_year      1052 non-null   int64   
 6   shape_area    350 non-null    float64 
 7   shape_length  350 non-null    float64 
 8   fid           190 non-null    float64 
 9   shape_area    96 non-null     float64 
 10  shape_len     96 non-null     float64 
 11  shape_leng    94 non-null     float64 
dtypes: float64(9), geometry(1), int64(1), object(1)
memory usage: 106.8+ KB


In [90]:
# Find all shape_area columns
shape_area = raw_gdf.loc[:, raw_gdf.columns == 'shape_area']

# Combine them, taking the first non-null value in each row
raw_gdf['shape_area_combined'] = shape_area.bfill(axis=1).iloc[:, 0]

# Drop the duplicate shape_area columns
raw_gdf = raw_gdf.drop(columns='shape_area')

# Rename the combined column
raw_gdf = raw_gdf.rename(columns={'shape_area_combined': 'shape_area'})

In [88]:
raw_gdf[raw_gdf.map_year == 2025].head()

,objectid,gang_name,shape_area,shape_length,geometry,map_year,shape_area,shape_length,fid,shape_area,shape_len,shape_leng
0,NaN,ambrose,1.266654e+07,NaN,"MULTIPOLYGON (((-87.67535 41.8487, -87.67575 4...",2025,2.120931e+06,12096.270557,1.0,NaN,NaN,29565.397156
1,NaN,bishops,8.713315e+06,NaN,"MULTIPOLYGON (((-87.67352 41.85952, -87.67308 ...",2025,1.460283e+06,6921.001557,2.0,NaN,NaN,16909.587419
2,NaN,black disciples,1.064841e+08,NaN,"MULTIPOLYGON (((-87.62323 41.69682, -87.62322 ...",2025,1.778474e+07,91834.606745,3.0,NaN,NaN,224632.140371
3,NaN,black p stones,1.844370e+08,NaN,"MULTIPOLYGON (((-87.65064 41.72138, -87.65062 ...",2025,3.083181e+07,130529.958812,4.0,NaN,NaN,319171.671612
4,NaN,black souls,1.419984e+07,NaN,"MULTIPOLYGON (((-87.70633 41.88772, -87.70632 ...",2025,2.381678e+06,14229.804591,5.0,NaN,NaN,34748.332135


In [91]:
top_gangs = raw_gdf[raw_gdf.map_year == 2025]\
            .nlargest(10, 'shape_area')\
            .gang_name.to_list()

top_gangs

['gangster disciples',
 'black p stones',
 'latin kings',
 'black disciples',
 'satan disciples',
 'two six',
 'four corner hustlers',
 'maniac latin disciples',
 'traveling vice lords',
 'conservative vice lords']

In [93]:
raw_gdf.gang_name.value_counts().head(20)

gang_name
ambrose                         19
spanish four corner hustlers    19
latin counts                    19
latin dragons                   19
latin eagles                    19
maniac latin disciples          19
mickey cobras                   19
milwaukee kings                 19
orchestra albany                19
pachucos                        19
party people                    19
satan disciples                 19
spanish gangster disciples      19
la raza                         19
unknown vice lords              19
undertaker vice lords           19
two-two boys                    19
traveling vice lords            19
new breed                       19
gangster disciples              19
Name: count, dtype: int64

In [ ]:
top_gangs_gdf = raw_gdf[raw_gdf.gang_name.isin(top_gangs)]
top_gangs_gdf

,objectid,gang_name,shape_length,geometry,map_year,shape_length,fid,shape_len,shape_leng,shape_area
7,8.0,black disciples,67061.870615,"MULTIPOLYGON (((-87.61637 41.7857, -87.61592 4...",2011,NaN,NaN,NaN,NaN,1.309874e+07
10,11.0,latin kings,135512.419791,"MULTIPOLYGON (((-87.68423 41.80129, -87.68423 ...",2011,NaN,NaN,NaN,NaN,4.115140e+07
32,33.0,maniac latin disciples,42769.850807,"MULTIPOLYGON (((-87.69833 41.9394, -87.69782 4...",2011,NaN,NaN,NaN,NaN,8.111832e+06
40,41.0,satan disciples,61074.108516,"MULTIPOLYGON (((-87.71873 41.81164, -87.71873 ...",2011,NaN,NaN,NaN,NaN,1.845934e+07
54,55.0,traveling vice lords,NaN,None,2011,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43,44.0,black disciples,92020.663952,"MULTIPOLYGON (((-87.62323 41.69682, -87.62322 ...",2018,NaN,NaN,NaN,NaN,1.619145e+07
44,45.0,black p stones,140319.108834,"MULTIPOLYGON (((-87.65064 41.72138, -87.65062 ...",2018,NaN,NaN,NaN,NaN,3.497617e+07
47,48.0,conservative vice lords,54290.017581,"MULTIPOLYGON (((-87.7433 41.81282, -87.74327 4...",2018,NaN,NaN,NaN,NaN,7.479311e+06
49,50.0,four corner hustlers,87384.036484,"MULTIPOLYGON (((-87.75472 41.90952, -87.75471 ...",2018,NaN,NaN,NaN,NaN,1.439948e+07


In [73]:
import folium
from folium.plugins import Timeline, TimelineSlider
from folium.features import GeoJsonPopup
from folium.utilities import JsCode

gdf = top_gangs_gdf[
    ["gang_name", "geometry", "map_year"]
].copy()

gdf = gdf.dropna(
    subset=["map_year", "gang_name"]
).copy()

gdf["map_year"] = gdf["map_year"].astype(int)

print("Years:", sorted(gdf["map_year"].unique()))
print("Gangs:", sorted(gdf["gang_name"].unique()))

gdf = gdf.to_crs(epsg=4326)

representative_points = gdf.geometry.representative_point()

center_lat = representative_points.y.mean()
center_lon = representative_points.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles=None
)

folium.TileLayer(
    tiles=(
        "https://basemaps.cartocdn.com/rastertiles/voyager/"
        "{z}/{x}/{y}.png"
        "?key=cb1_3t1q_1_0960046504f90cb9125e4501"
    ),
    attr="© OpenStreetMap contributors © CARTO",
    name="CartoDB Voyager",
    overlay=False,
    control=True,
    show=True
).add_to(m)

gang_names = sorted(gdf["gang_name"].unique())

colors = [
    "#e41a1c",
    "#377eb8",
    "#4daf4a",
    "#984ea3",
    "#ff7f00",
    "#a65628",
    "#f781bf",
    "#999999",
    "#66c2a5",
    "#fc8d62",
    "#8da0cb",
    "#e78ac3",
    "#a6d854",
    "#ffd92f",
    "#e5c494",
    "#b3b3b3",
    "#1b9e77",
    "#d95f02",
    "#7570b3",
    "#e7298a"
]

gang_colors = {
    gang: colors[i % len(colors)]
    for i, gang in enumerate(gang_names)
}

timelines = []

for gang in gang_names:

    gang_gdf = gdf[
        gdf["gang_name"] == gang
    ].copy()

    gang_geojson = gang_gdf.__geo_interface__

    for feature in gang_geojson["features"]:
        year = feature["properties"]["map_year"]

        feature["properties"]["start"] = f"{year}-01-01"
        feature["properties"]["end"] = f"{year}-12-31"

        feature["properties"]["gang_color"] = gang_colors[gang]

    gang_color = gang_colors[gang]

    timeline_style = JsCode(
        f"""
        function(data) {{
            return {{
                stroke: true,
                color: "{gang_color}",
                weight: 2,
                fillColor: "{gang_color}",
                fillOpacity: 0.50
            }};
        }}
        """
    )

    gang_group = folium.FeatureGroup(
        name=str(gang),
        overlay=True,
        show=True
    )

    gang_group.add_to(m)

    timeline = Timeline(
        gang_geojson,
        name=str(gang),
        style=timeline_style,
        overlay=True,
        control=False,
        show=True
    )

    timeline.add_to(gang_group)

    GeoJsonPopup(
        fields=["gang_name", "map_year"],
        aliases=["Gang:", "Year:"],
        labels=True,
        sticky=False
    ).add_to(timeline)

    timelines.append(timeline)

TimelineSlider(
    auto_play=False,
    show_ticks=True,
    enable_keyboard_controls=True,
    playback_duration=8000,
    date_options="YYYY"
).add_timelines(*timelines).add_to(m)

folium.LayerControl(
    collapsed=False
).add_to(m)

filename = "top_gangs_territority"

output_path = f"{viz_dir}/{filename}.html"

m.save(output_path)

print(f"Saved to: {output_path}")

Years: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Gangs: ['black disciples', 'black p stones', 'conservative vice lords', 'four corner hustlers', 'gangster disciples', 'latin kings', 'maniac latin disciples', 'satan disciples', 'traveling vice lords', 'two six']
Saved to: /Users/kchen/Documents/CMP_Projects/GangMaps/visualizations/top_gangs_territority.html


In [82]:
import folium
from folium.plugins import Timeline, TimelineSlider
from folium.features import GeoJsonPopup
from folium.utilities import JsCode

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

gdf = top_gangs_gdf[
    ["gang_name", "geometry", "map_year"]
].copy()

gdf = gdf.dropna(
    subset=["map_year", "gang_name"]
).copy()

gdf["map_year"] = gdf["map_year"].astype(int)

print("Years:", sorted(gdf["map_year"].unique()))
print("Gangs:", sorted(gdf["gang_name"].unique()))

# ---------------------------------------------------------
# Convert to WGS84
# ---------------------------------------------------------

gdf = gdf.to_crs(epsg=4326)

# ---------------------------------------------------------
# Map center
# ---------------------------------------------------------

representative_points = gdf.geometry.representative_point()

center_lat = representative_points.y.mean()
center_lon = representative_points.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles=None
)

# ---------------------------------------------------------
# Basemap
# ---------------------------------------------------------

folium.TileLayer(
    tiles=(
        "https://basemaps.cartocdn.com/rastertiles/light_all/"
        "{z}/{x}/{y}.png"
        "?key=cb1_3t1q_1_0960046504f90cb9125e4501"
    ),
    attr="© OpenStreetMap contributors © CARTO",
    name="Chicago Gangland",
    overlay=False,
    control=True,
    show=True
).add_to(m)

# ---------------------------------------------------------
# Gang colors
# ---------------------------------------------------------

gang_names = sorted(gdf["gang_name"].unique())

colors = [
    "#e41a1c",
    "#377eb8",
    "#4daf4a",
    "#984ea3",
    "#ff7f00",
    "#a65628",
    "#f781bf",
    "#999999",
    "#66c2a5",
    "#fc8d62",
    "#8da0cb",
    "#e78ac3",
    "#a6d854",
    "#ffd92f",
    "#e5c494",
    "#b3b3b3",
    "#1b9e77",
    "#d95f02",
    "#7570b3",
    "#e7298a"
]

gang_colors = {
    gang: colors[i % len(colors)]
    for i, gang in enumerate(gang_names)
}

# ---------------------------------------------------------
# Create timelines
# ---------------------------------------------------------

timelines = []

for gang in gang_names:

    gang_gdf = gdf[
        gdf["gang_name"] == gang
    ].copy()

    gang_geojson = gang_gdf.__geo_interface__

    # Add timeline dates
    for feature in gang_geojson["features"]:

        year = feature["properties"]["map_year"]

        feature["properties"]["start"] = f"{year}-01-01"
        feature["properties"]["end"] = f"{year}-12-31"

    gang_color = gang_colors[gang]

    # Timeline styling
    timeline_style = JsCode(
        f"""
        function(data) {{
            return {{
                stroke: true,
                color: "{gang_color}",
                weight: 2,
                fillColor: "{gang_color}",
                fillOpacity: 0.50
            }};
        }}
        """
    )

    # Gang layer
    gang_group = folium.FeatureGroup(
        name=str(gang),
        overlay=True,
        show=True
    )

    gang_group.add_to(m)

    # Timeline
    timeline = Timeline(
        gang_geojson,
        name=str(gang),
        style=timeline_style,
        overlay=True,
        control=False,
        show=True
    )

    timeline.add_to(gang_group)

    # Popup
    GeoJsonPopup(
        fields=["gang_name", "map_year"],
        aliases=["Gang:", "Year:"],
        labels=True,
        sticky=False
    ).add_to(timeline)

    timelines.append(timeline)

# ---------------------------------------------------------
# Timeline slider
# ---------------------------------------------------------

TimelineSlider(
    auto_play=False,
    show_ticks=True,
    enable_keyboard_controls=True,
    playback_duration=15000,
    date_options="YYYY"
).add_timelines(*timelines).add_to(m)

# ---------------------------------------------------------
# Layer control
# ---------------------------------------------------------

folium.LayerControl(
    collapsed=False
).add_to(m)

# ---------------------------------------------------------
# Layer control CSS
# ---------------------------------------------------------

layer_control_css = """
<style>

.leaflet-control-layers {
    font-size: 16px !important;
    line-height: 1.6 !important;
    min-width: 280px !important;
    max-height: 600px !important;
    overflow-y: auto !important;
}

.leaflet-control-layers-list {
    padding: 8px 12px !important;
}

.leaflet-control-layers-overlays label,
.leaflet-control-layers-base label {
    margin-bottom: 6px !important;
}

.leaflet-control-layers-selector {
    width: 18px !important;
    height: 18px !important;
    margin-right: 8px !important;
}

.gang-layer-button {
    display: inline-block;
    width: 48%;
    margin-top: 10px;
    padding: 8px 5px;
    background-color: white;
    border: 1px solid #999;
    border-radius: 4px;
    font-size: 14px;
    font-weight: bold;
    cursor: pointer;
    box-sizing: border-box;
}

.gang-layer-button:hover {
    background-color: #f0f0f0;
}

.gang-layer-button + .gang-layer-button {
    margin-left: 2%;
}

</style>
"""

m.get_root().html.add_child(
    folium.Element(layer_control_css)
)

# ---------------------------------------------------------
# Select / Deselect all JavaScript
# ---------------------------------------------------------

select_deselect_js = """
<script>

function setAllGangs(selected) {

    var layerControl = document.querySelector(
        '.leaflet-control-layers-overlays'
    );

    if (!layerControl) {
        return;
    }

    var checkboxes = layerControl.querySelectorAll(
        'input[type="checkbox"]'
    );

    checkboxes.forEach(function(checkbox) {

        if (checkbox.checked !== selected) {
            checkbox.click();
        }

    });
}

function selectAllGangs() {
    setAllGangs(true);
}

function deselectAllGangs() {
    setAllGangs(false);
}

</script>
"""

m.get_root().html.add_child(
    folium.Element(select_deselect_js)
)

# ---------------------------------------------------------
# Add Select All / Deselect All buttons
# ---------------------------------------------------------

button_html = """
<script>

document.addEventListener("DOMContentLoaded", function() {

    var interval = setInterval(function() {

        var overlayContainer = document.querySelector(
            '.leaflet-control-layers-overlays'
        );

        if (!overlayContainer) {
            return;
        }

        // Prevent duplicate buttons
        if (document.getElementById("gang-layer-buttons")) {
            clearInterval(interval);
            return;
        }

        // Button container
        var buttonContainer = document.createElement("div");

        buttonContainer.id = "gang-layer-buttons";
        buttonContainer.style.width = "100%";
        buttonContainer.style.display = "flex";
        buttonContainer.style.marginTop = "5px";

        // Select All button
        var selectButton = document.createElement("button");

        selectButton.id = "select-all-gangs";
        selectButton.type = "button";
        selectButton.className = "gang-layer-button";
        selectButton.innerHTML = "Select All Gangs";

        selectButton.onclick = function() {
            selectAllGangs();
        };

        // Deselect All button
        var deselectButton = document.createElement("button");

        deselectButton.id = "deselect-all-gangs";
        deselectButton.type = "button";
        deselectButton.className = "gang-layer-button";
        deselectButton.innerHTML = "Deselect All Gangs";

        deselectButton.onclick = function() {
            deselectAllGangs();
        };

        // Add buttons
        buttonContainer.appendChild(selectButton);
        buttonContainer.appendChild(deselectButton);

        overlayContainer.appendChild(buttonContainer);

        clearInterval(interval);

    }, 100);

});

</script>
"""

m.get_root().html.add_child(
    folium.Element(button_html)
)

# ---------------------------------------------------------
# Save map
# ---------------------------------------------------------

filename = "top_gangs_territority"

output_path = f"{viz_dir}/{filename}.html"

m.save(output_path)

print(f"Saved to: {output_path}")

Years: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Gangs: ['black disciples', 'black p stones', 'conservative vice lords', 'four corner hustlers', 'gangster disciples', 'latin kings', 'maniac latin disciples', 'satan disciples', 'traveling vice lords', 'two six']
Saved to: /Users/kchen/Documents/CMP_Projects/GangMaps/visualizations/top_gangs_territority.html
